In [13]:
import geopandas as gpd
import time

points_gdf=gpd.read_file(r'C:\Users\darey\OneDrive\Documents\MGEO\MGEO\BigGeospatilData\Task1\utrecht\point.shp')
lines_gdf=gpd.read_file(r'C:\Users\darey\OneDrive\Documents\MGEO\MGEO\BigGeospatilData\Task1\utrecht\line.shp')
extent_gdf=gpd.read_file(r'C:\Users\darey\OneDrive\Documents\MGEO\MGEO\BigGeospatilData\Task1\utrecht\extent.shp')


# --- Timing decorator ---------------------------------------------------------
def timed(func):
    """Decorator to print how long a geoprocessing operation takes."""
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        print(f"{func.__name__} took {t1 - t0:.3f} seconds")
        return result
    return wrapper


# --- Basic operations ---------------------------------------------------------
@timed
def make_buffer(gdf, distance=0.001):
    """
    Buffer all geometries in a GeoDataFrame.
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    out = gdf.copy()
    out["geometry"] = out.geometry.buffer(distance)
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)

@timed
def make_convex_hull(gdf):
    """
    Convex hull of ALL features together (one polygon result).
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    hull_geom = gdf.unary_union.convex_hull
    out = gpd.GeoDataFrame(geometry=[hull_geom], crs=gdf.crs)
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)

@timed
def dissolve_by(gdf, field):
    """
    Dissolve features using a field.
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    out = gdf.dissolve(by=field)
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)


# --- Overlay & Clip operations -----------------------------------------------
@timed
def clip_layer(target_gdf, clip_gdf):
    """
    Clip target_gdf with clip_gdf.
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    out = gpd.clip(target_gdf, clip_gdf)
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)

@timed
def difference_layer(gdf1, gdf2):
    """
    Difference: gdf1 minus gdf2.
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    out = gpd.overlay(gdf1, gdf2, how="difference")
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)

@timed
def intersection_layer(gdf1, gdf2):
    """
    Intersection of two layers.
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    out = gpd.overlay(gdf1, gdf2, how="intersection")
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)

@timed
def union_layer(gdf1, gdf2):
    """
    Union of two layers.
    Returns (output_gdf, time_in_seconds)
    """
    t0 = time.perf_counter()
    out = gpd.overlay(gdf1, gdf2, how="union")
    t1 = time.perf_counter()
    return out, round(t1 - t0, 4)

In [16]:
extent_gdf.columns

Index(['fid', 'gid', 'id', 'country', 'name', 'enname', 'locname', 'offname',
       'boundary', 'adminlevel', 'wikidata', 'wikimedia', 'timestamp', 'note',
       'path', 'rpath', 'iso3166_2', 'geometry'],
      dtype='object')

In [ ]:
# ---- BUFFER -----------------------------------------------------------------
print('Buffer - Point')
buffer_point, t_buffer_point = make_buffer(points_gdf)
print("Time:", t_buffer_point, "sec\n")

print('Buffer - Line')
buffer_line, t_buffer_line = make_buffer(lines_gdf)
print("Time:", t_buffer_line, "sec\n")

print('Buffer - Extent')
buffer_extent, t_buffer_extent = make_buffer(extent_gdf)
print("Time:", t_buffer_extent, "sec\n")


# ---- CONVEX HULL ------------------------------------------------------------
print('Convex Hull - Point')
hull_point, t_hull_point = make_convex_hull(points_gdf)
print("Time:", t_hull_point, "sec\n")

print('Convex Hull - Line')
hull_line, t_hull_line = make_convex_hull(lines_gdf)
print("Time:", t_hull_line, "sec\n")

print('Convex Hull - Extent')
hull_extent, t_hull_extent = make_convex_hull(extent_gdf)
print("Time:", t_hull_extent, "sec\n")


# ---- DISSOLVE ---------------------------------------------------------------
print('Dissolve - Point')
dissolve_point, t_dissolve_point = dissolve_by(points_gdf, 'postcode')
print("Time:", t_dissolve_point, "sec\n")

print('Dissolve - Line')
dissolve_line, t_dissolve_line = dissolve_by(lines_gdf, 'gme_naam')
print("Time:", t_dissolve_line, "sec\n")

print('Dissolve - Extent')
dissolve_extent, t_dissolve_extent = dissolve_by(extent_gdf, 'offname')
print("Time:", t_dissolve_extent, "sec\n")


# ---- CLIP -------------------------------------------------------------------
print('Clip - Point (Point Buffer clipped by Line Buffer)')
clip_point_line, t_clip_point_line = clip_layer(buffer_point, buffer_line)
print("Time:", t_clip_point_line, "sec\n")

print('Clip - Line (Line Buffer clipped by Extent Buffer)')
clip_line_extent, t_clip_line_extent = clip_layer(buffer_line, buffer_extent)
print("Time:", t_clip_line_extent, "sec\n")


Buffer - Point


C:\Users\darey\AppData\Local\Temp\ipykernel_39820\555071921.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  out["geometry"] = out.geometry.buffer(distance)


make_buffer took 7.277 seconds
Time: 7.2766 sec

Buffer - Line
make_buffer took 3.052 seconds
Time: 3.0521 sec

Buffer - Extent
make_buffer took 0.068 seconds
Time: 0.0683 sec

Convex Hull - Point


C:\Users\darey\AppData\Local\Temp\ipykernel_39820\555071921.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hull_geom = gdf.unary_union.convex_hull


make_convex_hull took 1.120 seconds
Time: 1.1203 sec

Convex Hull - Line


C:\Users\darey\AppData\Local\Temp\ipykernel_39820\555071921.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hull_geom = gdf.unary_union.convex_hull


make_convex_hull took 1.074 seconds
Time: 1.0745 sec

Convex Hull - Extent
make_convex_hull took 0.008 seconds
Time: 0.0077 sec

Dissolve - Point


C:\Users\darey\AppData\Local\Temp\ipykernel_39820\555071921.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hull_geom = gdf.unary_union.convex_hull


dissolve_by took 2.284 seconds
Time: 2.2844 sec

Dissolve - Line
dissolve_by took 1.048 seconds
Time: 1.0478 sec

Dissolve - Extent
dissolve_by took 0.009 seconds
Time: 0.0089 sec

Clip - Point (Point Buffer clipped by Line Buffer)


In [ ]:
# ---- DIFFERENCE --------------------------------------------------------------
print('Difference - Point Buffer minus Line Buffer')
diff_point_line, t_diff_point_line = difference_layer(buffer_point, buffer_line)
print("Time:", t_diff_point_line, "sec\n")

print('Difference - Line Buffer minus Extent Buffer')
diff_line_extent, t_diff_line_extent = difference_layer(buffer_line, buffer_extent)
print("Time:", t_diff_line_extent, "sec\n")


# ---- INTERSECTION ------------------------------------------------------------
print('Intersection - Point Buffer ∩ Line Buffer')
inter_point_line, t_inter_point_line = intersection_layer(buffer_point, buffer_line)
print("Time:", t_inter_point_line, "sec\n")

print('Intersection - Line Buffer ∩ Extent Buffer')
inter_line_extent, t_inter_line_extent = intersection_layer(buffer_line, buffer_extent)
print("Time:", t_inter_line_extent, "sec\n")


# ---- UNION -------------------------------------------------------------------
print('Union - Point Buffer ∪ Line Buffer')
union_point_line, t_union_point_line = union_layer(buffer_point, buffer_line)
print("Time:", t_union_point_line, "sec\n")

print('Union - Line Buffer ∪ Extent Buffer')
union_line_extent, t_union_line_extent = union_layer(buffer_line, buffer_extent)
print("Time:", t_union_line_extent, "sec\n")

In [ ]:
def calculate_area(gdf):
    return gdf.geometry.area    